# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nnanwubeikenna-prog/ikenna-flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [15]:
import duckdb
import pandas as pd
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{hf_token}')")

query = """
SELECT
    content_hash_id,
        client_hash_id,
            content_type,
                COALESCE(search_volume, 0) AS search_volume,
                    COALESCE(word_count, 0) AS word_count,
                        COALESCE(backlinks, 0) AS backlinks,
                            COALESCE(competition, 0.0) AS competition,
                                CASE WHEN content_updated_date < DATE '2025-06-01' OR content_updated_date IS NULL THEN 1 ELSE 0 END AS is_stale,
                                    CASE WHEN (content_updated_date < DATE '2025-06-01' OR content_updated_date IS NULL) OR (word_count < 800 OR word_count IS NULL) THEN 1 ELSE 0 END AS target_decay
                                    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet')
                                    WHERE is_published = TRUE AND is_deleted = FALSE;
"""
df = con.sql(query).df()
df_features = pd.get_dummies(df, columns=['content_type'], drop_first=True)
print(f"Cohort size: {len(df_features)} across {df['client_hash_id'].nunique()} unique clients.")
print(df_features.head(3))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Cohort size: 411540 across 72 unique clients.
            content_hash_id           client_hash_id  search_volume  \
0  content_004de9653278b5a4  client_04660893ae39614a             30   
1  content_00dc5efae381b2ab  client_04660893ae39614a             10   
2  content_01410f2556c327ac  client_04660893ae39614a            480   

   word_count  backlinks  competition  is_stale  target_decay  \
0        2555         16         0.91         0             0   
1        2430          0         0.00         0             0   
2        2645        169         0.36         0             0   

   content_type_feedly article  content_type_keyword article  
0                        False                          True  
1                        False                          True  
2                        False                          True  


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [16]:
import pandas as pd
feature_summary = pd.DataFrame([
            {"Feature": "search_volume", "Meaning": "Monthly organic search demand", "Missing Strategy": "Imputed 0", "Available Pre-Prediction": "Yes"},
                {"Feature": "word_count", "Meaning": "Total published word count", "Missing Strategy": "Imputed 0", "Available Pre-Prediction": "Yes"},
                    {"Feature": "backlinks", "Meaning": "Inbound external referring domains", "Missing Strategy": "Imputed 0", "Available Pre-Prediction": "Yes"},
                        {"Feature": "competition", "Meaning": "Keyword competition index (0.0 - 1.0)", "Missing Strategy": "Imputed 0.0", "Available Pre-Prediction": "Yes"},
                            {"Feature": "is_stale", "Meaning": "Flag for update date prior to 2025-06-01", "Missing Strategy": "Treated as stale (1)", "Available Pre-Prediction": "Yes"}
])
print(feature_summary.to_string(index=False))

      Feature                                  Meaning     Missing Strategy Available Pre-Prediction
search_volume            Monthly organic search demand            Imputed 0                      Yes
   word_count               Total published word count            Imputed 0                      Yes
    backlinks       Inbound external referring domains            Imputed 0                      Yes
  competition    Keyword competition index (0.0 - 1.0)          Imputed 0.0                      Yes
     is_stale Flag for update date prior to 2025-06-01 Treated as stale (1)                      Yes


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [17]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import KFold, GroupKFold, cross_val_score

base_rate = df_features['target_decay'].mean()
print(f"Target Base Rate: {base_rate:.4f} ({base_rate * 100:.2f}%)")

forbidden = ['trend_direction', 'trend_pct', 'post_refresh_clicks']
assert not any(col in df_features.columns for col in forbidden), "Forbidden columns detected!"
print("Leakage Test: PASSED (0 future/leaky columns)")

feature_cols = [c for c in df_features.columns if c not in ['content_hash_id', 'client_hash_id', 'target_decay']]
X = df_features[feature_cols]
y = df_features['target_decay']
groups = df_features['client_hash_id']

clf = RandomForestClassifier(n_estimators=50, max_depth=6, random_state=42, n_jobs=-1)

random_auc = cross_val_score(clf, X, y, cv=KFold(n_splits=5, shuffle=True, random_state=42), scoring='roc_auc').mean()
grouped_auc = cross_val_score(clf, X, y, groups=groups, cv=GroupKFold(n_splits=5), scoring='roc_auc').mean()

print(f"Random 5-Fold ROC-AUC:  {random_auc:.4f}")
print(f"Grouped 5-Fold ROC-AUC: {grouped_auc:.4f}")
print(f"Memorization Gap:       {random_auc - grouped_auc:.4f}")

Target Base Rate: 0.2841 (28.41%)
Leakage Test: PASSED (0 future/leaky columns)
Random 5-Fold ROC-AUC:  1.0000
Grouped 5-Fold ROC-AUC: 1.0000
Memorization Gap:       0.0000


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [18]:
import pandas as pd
exclusions = pd.DataFrame([
            {"Excluded Field": "trend_direction / trend_pct", "Reason": "Future outcome leakage (measures post-period traffic decay)"},
                {"Excluded Field": "client_hash_id / content_hash_id", "Reason": "Entity memorization risk; reserved exclusively for GroupKFold"},
                    {"Excluded Field": "is_published / is_deleted", "Reason": "Zero-variance constants after cohort filtering"},
                        {"Excluded Field": "Raw URLs / Client Names", "Reason": "Strict privacy safety and public dataset governance"}
])
print(exclusions.to_string(index=False))

                  Excluded Field                                                        Reason
     trend_direction / trend_pct   Future outcome leakage (measures post-period traffic decay)
client_hash_id / content_hash_id Entity memorization risk; reserved exclusively for GroupKFold
       is_published / is_deleted                Zero-variance constants after cohort filtering
         Raw URLs / Client Names           Strict privacy safety and public dataset governance


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.